In [1]:
import sumo_rl
import numpy as np

# Initialize raw PettingZoo environment
env = sumo_rl.parallel_env(
    net_file="./env/3x3Grid2lanes.net.xml",
    route_file="./env/routes14000.rou.xml",
    use_gui=True, # Keep False for training
    num_seconds=3600,
    delta_time=5,
    reward_fn="pressure",
)

# Get dimensions for the network
sample_agent = env.possible_agents[0]
obs_dim = env.observation_space(sample_agent).shape[0]
action_dim = env.action_space(sample_agent).n

 Retrying in 1 seconds
Step #0.00 (0ms ?*RT. ?UPS, TraCI: 30ms, vehicles TOT 0 ACT 0 BUF 0)                     


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

class QNetwork(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(obs_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [3]:
# Hyperparameters
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
batch_size = 64
learning_rate = 1e-3

In [4]:

# Initialize Network, Optimizer, and Buffer
q_net = QNetwork(obs_dim, action_dim)
optimizer = optim.Adam(q_net.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()
replay_buffer = deque(maxlen=10000)



In [ ]:
episodes = 50

for episode in range(episodes):
    observations, infos = env.reset()
    episode_reward = 0
    
    while env.agents:
        actions = {}
        
        # --- 1. Epsilon-Greedy Action Selection ---
        for agent in env.agents:
            obs_tensor = torch.FloatTensor(observations[agent]).unsqueeze(0)
            
            if random.random() < epsilon:
                # Explore
                actions[agent] = env.action_space(agent).sample()
            else:
                # Exploit
                with torch.no_grad():
                    q_values = q_net(obs_tensor)
                    actions[agent] = torch.argmax(q_values).item()

        # --- 2. Step the Environment ---
        next_observations, rewards, terminations, truncations, infos = env.step(actions)

        # --- 3. Store Transitions in Shared Buffer ---
        for agent in env.agents:
            replay_buffer.append((
                observations[agent], 
                actions[agent], 
                rewards[agent], 
                next_observations[agent], 
                terminations[agent] or truncations[agent]
            ))
            episode_reward += rewards[agent]

        observations = next_observations

        # --- 4. Train the Q-Network ---
        if len(replay_buffer) > batch_size:
            # Sample a batch
            batch = random.sample(replay_buffer, batch_size)
            b_obs, b_act, b_rew, b_next_obs, b_done = zip(*batch)
            
            b_obs = torch.FloatTensor(np.array(b_obs))
            b_act = torch.LongTensor(b_act).unsqueeze(1)
            b_rew = torch.FloatTensor(b_rew).unsqueeze(1)
            b_next_obs = torch.FloatTensor(np.array(b_next_obs))
            b_done = torch.FloatTensor(b_done).unsqueeze(1)

            # Compute Current Q values
            current_q = q_net(b_obs).gather(1, b_act)
            
            # Compute Target Q values (Note: A separate Target Network is recommended for stability)
            with torch.no_grad():
                max_next_q = q_net(b_next_obs).max(1)[0].unsqueeze(1)
                target_q = b_rew + (gamma * max_next_q * (1 - b_done))
            
            # Optimize
            loss = loss_fn(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Decay Epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    print(f"Episode {episode + 1} | Total Shared Reward: {episode_reward:.2f} | Epsilon: {epsilon:.2f}")

env.close()

 Retrying in 1 seconds
Episode 1 | Total Shared Reward: -33418.00 | Epsilon: 0.99
 Retrying in 1 seconds
Episode 2 | Total Shared Reward: -17370.00 | Epsilon: 0.99
 Retrying in 1 seconds
Episode 3 | Total Shared Reward: -15691.00 | Epsilon: 0.99
 Retrying in 1 seconds
Episode 4 | Total Shared Reward: -14269.00 | Epsilon: 0.98
 Retrying in 1 seconds
Episode 5 | Total Shared Reward: -15738.00 | Epsilon: 0.98
 Retrying in 1 seconds
Episode 6 | Total Shared Reward: -14414.00 | Epsilon: 0.97
 Retrying in 1 seconds
Episode 7 | Total Shared Reward: -31637.00 | Epsilon: 0.97
 Retrying in 1 seconds
Episode 8 | Total Shared Reward: -14405.00 | Epsilon: 0.96
 Retrying in 1 seconds
Episode 9 | Total Shared Reward: -14280.00 | Epsilon: 0.96
 Retrying in 1 seconds
Episode 10 | Total Shared Reward: -17011.00 | Epsilon: 0.95
 Retrying in 1 seconds
Episode 11 | Total Shared Reward: -13497.00 | Epsilon: 0.95
 Retrying in 1 seconds
Episode 12 | Total Shared Reward: -14003.00 | Epsilon: 0.94
 Retrying in 

In [ ]:
torch.save(q_net.state_dict(), "3x3.pth")